# 04 — Event study: abnormal returns around the April 2, 2025 tariff announcement

**Timing.** The announcement came **after the close on Wed Apr 2, 2025**, so **day 0 = Thu Apr 3**. The 90-day pause came midday on **Wed Apr 9 (day +4)**, so no window used here goes past day +3.

**Market model** (estimated over trading days −260 to −11, about one year, ending two weeks before the event so the event can't leak into the estimate):

$$R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}$$

**Abnormal return** on event day *t*, and **cumulative abnormal return** over a window:

$$AR_{i,t} = R_{i,t} - (\hat\alpha_i + \hat\beta_i R_{m,t}) \qquad CAR_i[t_1,t_2] = \sum_{t=t_1}^{t_2} AR_{i,t}$$

| Column | Window | Dates |
|---|---|---|
| `car_0_1` (**main**) | [0, +1] | Apr 3 – Apr 4 |
| `car_m1_1` | [−1, +1] | Apr 2 – Apr 4 |
| `car_0_3` | [0, +3] | Apr 3 – Apr 8 |

We also report the **market-adjusted** return (stock minus SPY, no regression) as a check that does not depend on beta estimates.

| | |
|---|---|
| **Inputs** | `data/raw/market_data/daily_prices.csv` |
| **Outputs** | `data/processed/event_car.csv` (one row per firm)<br>`data/processed/daily_abnormal_returns.csv` (daily AR paths for the dashboard) |


In [ ]:
# --- Setup: find the repo root and load shared settings from code/config.py ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Can't find the repo root. Run this from inside the project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))

import config as cfg
print("Repo root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd

prices = pd.read_csv(cfg.PRICES_CSV, parse_dates=["date"])
adj = prices.pivot(index="date", columns="ticker", values="adj_close").sort_index()
ret = adj.pct_change(fill_method=None).iloc[1:]     # simple daily returns

dates = ret.index
pos0 = int(np.searchsorted(dates, pd.Timestamp(cfg.EVENT_DAY0)))
assert dates[pos0] == pd.Timestamp(cfg.EVENT_DAY0), f"day 0 should be {cfg.EVENT_DAY0}, got {dates[pos0].date()}"
assert pos0 + cfg.EST_START >= 0, "price history starts too late for the estimation window: move PRICE_START earlier"

def day(k):
    """Calendar date of event day k."""
    return dates[pos0 + k].date()

print("Estimation window:", day(cfg.EST_START), "to", day(cfg.EST_END))
for name, (a, b) in cfg.CAR_WINDOWS.items():
    print(f"{name}: {day(a)} to {day(b)}")

## Estimate the market model and compute CARs

In [ ]:
m = cfg.MARKET
rows, ar_paths, problems = [], [], []

for t in [c for c in ret.columns if c != m]:
    est = ret.iloc[pos0 + cfg.EST_START: pos0 + cfg.EST_END + 1][[t, m]].dropna()
    if len(est) < cfg.MIN_EST_OBS:
        problems.append({"ticker": t, "step": "04_car", "problem": f"only {len(est)} estimation days"})
        continue

    beta, alpha = np.polyfit(est[m], est[t], 1)          # OLS slope and intercept
    resid = est[t] - (alpha + beta * est[m])
    ar = ret[t] - (alpha + beta * ret[m])                # abnormal return, every day

    row = {"ticker": t, "alpha": alpha, "beta": beta, "est_obs": len(est),
           "resid_sd": resid.std(ddof=2),
           "pre_event_vol": ret[t].iloc[pos0 - cfg.VOL_WINDOW: pos0].std()}
    for name, (a, b) in cfg.CAR_WINDOWS.items():
        row[name] = ar.iloc[pos0 + a: pos0 + b + 1].sum()
        row[name.replace("car", "mktadj")] = (ret[t] - ret[m]).iloc[pos0 + a: pos0 + b + 1].sum()
    rows.append(row)

    lo, hi = cfg.AR_PATH_WINDOW
    for k in range(lo, hi + 1):
        ar_paths.append({"ticker": t, "event_day": k, "date": day(k), "ar": ar.iloc[pos0 + k]})

car = pd.DataFrame(rows)
car.to_csv(cfg.CAR_CSV, index=False)

path = pd.DataFrame(ar_paths)
path["car_cum"] = path.groupby("ticker")["ar"].cumsum()
path.to_csv(cfg.DAILY_AR_CSV, index=False)

if problems:
    probs = pd.read_csv(cfg.DOWNLOAD_PROBLEMS_CSV)
    pd.concat([probs, pd.DataFrame(problems)]).to_csv(cfg.DOWNLOAD_PROBLEMS_CSV, index=False)

car.round(4)

## Checks
- No missing returns on event days.
- Betas in a sane range (roughly 0–3 for large US stocks).
- The market-model CAR and market-adjusted CAR should be close. A big gap means the beta estimate is driving the result.

In [ ]:
assert car[list(cfg.CAR_WINDOWS)].notna().all().all(), "missing CAR values"
odd_beta = car[(car["beta"] < 0) | (car["beta"] > 3)]
if len(odd_beta):
    print("WARNING: unusual betas:\n", odd_beta[["ticker", "beta"]])
print("Correlation, market-model CAR vs market-adjusted CAR [0,+1]:",
      round(car["car_0_1"].corr(car["mktadj_0_1"]), 3))

## Check by hand: recompute one firm's CAR a different way
Start again from the raw price file, use a different regression routine (`np.linalg.lstsq` instead of `np.polyfit`), and look the dates up by calendar date instead of by position. It must match the table above.

In [ ]:
t = car["ticker"].iloc[0]
p = pd.read_csv(cfg.PRICES_CSV, parse_dates=["date"])
w = p.pivot(index="date", columns="ticker", values="adj_close").sort_index()
r = (w / w.shift(1) - 1).dropna(how="all")
d = list(r.index)
i0 = d.index(pd.Timestamp(cfg.EVENT_DAY0))
est = r.iloc[i0 + cfg.EST_START: i0 + cfg.EST_END + 1][[t, cfg.MARKET]].dropna()
X = np.column_stack([np.ones(len(est)), est[cfg.MARKET]])
a_hat, b_hat = np.linalg.lstsq(X, est[t], rcond=None)[0]
window = r.loc[pd.Timestamp("2025-04-03"):pd.Timestamp("2025-04-04")]
hand_car = (window[t] - (a_hat + b_hat * window[cfg.MARKET])).sum()

pipeline_car = car.set_index("ticker").loc[t, "car_0_1"]
print(f"{t}: hand CAR[0,+1] = {hand_car:.6f}   pipeline = {pipeline_car:.6f}")
assert np.isclose(hand_car, pipeline_car), "hand check FAILED"
print("Hand check passed.")